In [ ]:
# -------------------------------
# 1. Load processed feature dataset for modelling
# -------------------------------

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df_features = pd.read_csv("../data/processed/df_features.csv")
print("Loaded dataset shape:", df_features.shape)
display(df_features.head())

In [ ]:
# Define features and targets
features = ["log_dbytes", "log_min_flow_bytes", "log_byte_prod", "Dload", "load_skew", "log_dmeansz", "mean_pkt_sz_ratio", "sttl", "dttl", "ttl_diff", "ct_state_ttl", "ackdat", "synack"]
X = df_features[features]
y = df_features["Label"]

print("Feature matrix shape:", X.shape)
print("Target vector shape:", y.shape)

In [ ]:
# -------------------------------
# 2. Fit isolation forest
# -------------------------------

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler
from sklearn.ensemble import IsolationForest
from sklearn.model_selection import train_test_split

df_train, df_test = train_test_split(
    df_features, test_size=0.2, stratify=df_features["Label"], random_state=42
)

X_train = df_train[features]
y_train = df_train["Label"]

X_test = df_test[features]
y_test = df_test["Label"]

attack_fraction = y_train.sum()/len(y_train)
print("Contamination fraction:", attack_fraction)

# Model initialisation
model = Pipeline([
    (
        "scaler",
        RobustScaler()
    ), (
        "iforest",
        IsolationForest(
            n_estimators=1500,
            max_samples=200,
            contamination=0.075,
            random_state=42,
            n_jobs=-1
        )
    )
])

# Model fit
model.fit(X_train)

In [ ]:
# Hyperparameter search
from sklearn.model_selection import StratifiedKFold
import itertools

# Hyperparameter grid
n_estimators_list = [800, 1000, 1200]
max_samples_list = [150, 200, 250]
contamination_factors = [0.8, 1, 1.2]

kf = StratifiedKFold(n_splits=4, shuffle=True, random_state=42)
seeds = [42, 101]  # ensemble seeds

results = []

for n, ms, cont in itertools.product(n_estimators_list, max_samples_list, contamination_factors):
    fold_f1s = []
    fold_precisions = []
    fold_recalls = []

    for train_idx, test_idx in kf.split(X, y):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

        attack_fraction = y_train.sum() / len(y_train) * cont

        # Ensemble predictions
        ensemble_preds = np.zeros(len(X_test))
        for seed in seeds:
            model = IsolationForest(
                n_estimators=n,
                max_samples=ms,
                contamination=attack_fraction,
                random_state=seed,
                n_jobs=-1
            )
            model.fit(X_train)

            y_pred = pd.Series(model.predict(X_test)).map({1:0, -1:1}).values
            ensemble_preds += y_pred

        # Majority vote
        y_pred_final = (ensemble_preds >= (len(seeds)/2)).astype(int)

        # Compute metrics
        fold_f1s.append(f1_score(y_test, y_pred_final))
        fold_precisions.append(precision_score(y_test, y_pred_final))
        fold_recalls.append(recall_score(y_test, y_pred_final))
    
    # Store mean metrics across folds
    results.append({
        "n_estimators": n,
        "max_samples": ms,
        "contamination_factor": cont,
        "f1_attack_mean": np.mean(fold_f1s),
        "f1_attack_std": np.std(fold_f1s),
        "precision_attack_mean": np.mean(fold_precisions),
        "recall_attack_mean": np.mean(fold_recalls)
    })

results_df = pd.DataFrame(results)
print(results_df.sort_values(by="f1_attack_mean", ascending=False).head(10))

In [ ]:
# -------------------------------
# 3. Generate predictions and anomaly scores
# -------------------------------
anomaly_scores = -model.decision_function(X_test)
pred_labels = model.predict(X_test)
pred_labels_binary = (pred_labels == -1).astype(int)

df_test["Label"] = y_test
df_test["anomaly_score"] = anomaly_scores
df_test["pred_anomaly"] = pred_labels_binary

# Quick peek
df_test[["Label", "anomaly_score", "pred_anomaly"]].head()

In [ ]:
from sklearn.metrics import precision_score, recall_score

# Manual thresholding
plt.hist(anomaly_scores, bins=100)
plt.xlabel("Anomaly Score")
plt.ylabel("Count")
plt.title("Isolation Forest Anomaly Scores")
plt.show()

percentiles = np.arange(95, 99.6, 0.1)
rows = []

y_test = y_test.reset_index(drop=True)
X_test = X_test.reset_index(drop=True)

for p in percentiles:
    threshold = np.percentile(anomaly_scores, p)
    y_pred = (anomaly_scores >= threshold).astype(int)
    rows.append({
        "percentile": p,
        "precision": precision_score(y_test, y_pred),
        "recall": recall_score(y_test, y_pred)
    })
pr_df = pd.DataFrame(rows)
display(pr_df)

plt.figure(figsize=(8, 6))
plt.plot(pr_df["recall"], pr_df["precision"], marker="o")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall via decision_function threshold")
plt.grid(True)
plt.show()

In [ ]:
# -------------------------------
# 4. Evaluate and visualise predictions
# -------------------------------

import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report

# Quick stats
num_anomalies = df_test["pred_anomaly"].sum()
print(f"Number of predicted anomalies: {num_anomalies}")

# Confusion matrix
cm = confusion_matrix(df_test["Label"], df_test["pred_anomaly"])
sns.heatmap(cm, annot=True, fmt='d', cmap="Blues", xticklabels=["Normal", "Anomaly"], yticklabels=["Normal", "Anomaly"])
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix: Isolation Forest vs True Labels")
plt.show()

# Classification report
print(classification_report(df_test["Label"], df_test["pred_anomaly"]))

In [ ]:
# Visualise anomalies in feature space
plt.figure(figsize=(8, 6))
sns.scatterplot(
    data=df_test,
    x="log_byte_ratio",
    y="log_dur",
    hue="Label",
    style="pred_anomaly",
    alpha=0.6,
    palette={0: "blue", 1: "red"}
)
plt.xlabel("Log(Byte Ratio)")
plt.ylabel("Log(Duration)")
plt.title("Predicted Anomalies vs True Attack Labels")
plt.show()

In [ ]:
# Create outcome categories
df_test["outcome"] = "TN" # default
df_test.loc[(df_test["Label"] == 1) & (df_test["pred_anomaly"] == 1), "outcome"] = "TP"
df_test.loc[(df_test["Label"] == 1) & (df_test["pred_anomaly"] == 0), "outcome"] = "FN"
df_test.loc[(df_test["Label"] == 0) & (df_test["pred_anomaly"] == 1), "outcome"] = "FP"
# Sanity check
display(df_test["outcome"].value_counts())

In [ ]:
attack_subset = df_test[df_test["Label"] == 1]
display(
    attack_subset.groupby("outcome")[
        [
            "log_byte_ratio",
            "log_dur",
            "log_sbytes",
            "log_dbytes",
        ]
    ].median()
)

In [ ]:
plt.figure(figsize=(8, 6))
sns.scatterplot(
    data=attack_subset,
    x="log_byte_ratio",
    y="log_dur",
    hue="outcome",
    palette={"TP": "red", "FN": "orange"},
    alpha=0.6
)
plt.xlabel("Log(Byte Ratio)")
plt.ylabel("Log(Duration)")
plt.title("Detected vs Missed Attacks (TP vs FN)")
plt.show()

In [ ]:
plt.figure(figsize=(8, 6))
sns.scatterplot(
    data=attack_subset,
    x="log_Sload",
    y="log_Dload",
    hue="outcome",
    palette={"TP": "red", "FN": "orange"},
    alpha=0.6
)
plt.xlabel("Log(Source Bytes per Duration)")
plt.ylabel("Log(Destination Bytes per Duration)")
plt.title("Detected vs Missed Attacks (TP vs FN) by Flow Rates")
plt.show()

In [ ]:
plt.figure(figsize=(8, 6))
attack_subset["avg_d_pkt_size"] = attack_subset["dbytes"]/attack_subset["Dpkts"].replace(0, np.nan)
attack_subset["log_avg_d_pkt_size"] = np.log1p(attack_subset["avg_d_pkt_size"])
attack_subset["log_avg_d_pkt_size"] = attack_subset["log_avg_d_pkt_size"].fillna(0)
sns.boxplot(
    x="outcome",
    y="log_avg_d_pkt_size",
    data=attack_subset
)
plt.title("Distribution of log_avg_d_pkt_size for TPs vs FNs")
plt.show()

In [ ]:
plt.figure(figsize=(8, 6))
sns.boxplot(
    x="outcome",
    y="d_to_s_pkt_ratio",
    data=attack_subset
)
plt.title("Distribution of d_to_s_pkt_ratio for TPs vs FNs")
plt.show()

In [ ]:
# Distinguish FPs from attack flows
fp_subset = df_test[df_test["outcome"] == "FP"]
attack_subset = df_test[df_test["Label"] == 1]

# Pick numeric features
numeric_features = [col for col in df_features.columns if df_features[col].dtype in [float, int]]

# Safe numeric subsets
fp_safe = fp_subset[numeric_features].replace([np.inf, -np.inf], np.nan).dropna()
attack_safe = attack_subset[numeric_features].replace([np.inf, -np.inf], np.nan).dropna()

def cohens_d(x, y):
    """Compute Cohen's d for two arrays."""
    nx = len(x)
    ny = len(y)
    dof = nx + ny - 2
    pooled_std = np.sqrt(((x.std(ddof=1) ** 2) * (nx - 1) + (y.std(ddof=1) ** 2) * (ny - 1)) / dof)
    if pooled_std == 0:
        return 0
    return (x.mean() - y.mean()) / pooled_std

effect_sizes = {}
for feat in numeric_features:
    effect_sizes[feat] = cohens_d(fp_safe[feat], attack_safe[feat])

effect_sizes_df = pd.DataFrame.from_dict(effect_sizes, orient='index', columns=['cohens_d'])
effect_sizes_df['abs_d'] = effect_sizes_df['cohens_d'].abs()
effect_sizes_df = effect_sizes_df.sort_values(by='abs_d', ascending=False)

print(effect_sizes_df.head(15))

In [ ]:
# Attacks vs normal flows
attack_subset = df_features[df_features["Label"] == 1]
normal_subset = df_features[df_features["Label"] == 0]

# Compute Cohen's d as before
effect_sizes_all = {}
for feat in numeric_features:
    effect_sizes_all[feat] = cohens_d(
        attack_subset[feat].replace([np.inf, -np.inf], np.nan).dropna(),
        normal_subset[feat].replace([np.inf, -np.inf], np.nan).dropna()
    )

effect_sizes_all_df = pd.DataFrame.from_dict(effect_sizes_all, orient='index', columns=['cohens_d'])
effect_sizes_all_df['abs_d'] = effect_sizes_all_df['cohens_d'].abs()
effect_sizes_all_df = effect_sizes_all_df.sort_values(by='abs_d', ascending=False)
print(effect_sizes_all_df.head(15))